# 📘 Fincept Notebook — Monte Carlo Price Simulation

**Quant · Intermediate · ~22 min · pandas + numpy**

Geometric Brownian Motion (GBM) is the workhorse model for simulating asset prices — it's the same diffusion that sits underneath the Black-Scholes formula. A price evolves with a deterministic **drift** (the trend) and a random **diffusion** (the noise), and the magic of Monte Carlo is that by simulating thousands of independent paths we can read off the *entire distribution* of where the price might land — not just a point forecast. From that distribution we extract percentile cones, Value at Risk, and shortfall probabilities.

**What you'll learn**
- The GBM equation and how drift vs diffusion shape a price path
- How to simulate thousands of daily-stepped paths with numpy and cumulate log returns
- How to read percentile cones, 95% Value at Risk, and the probability of ending below the start

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. The model and parameters

Under GBM the log return over a small time step `dt` is normally distributed:

$$\Delta \ln S \sim \mathcal{N}\big((\mu - \tfrac{1}{2}\sigma^2)\,dt,\;\; \sigma^2\,dt\big)$$

- **μ** — annual drift (expected continuously-compounded return).
- **σ** — annual volatility (diffusion).
- The `−½σ²` term is the **Itô correction**: because of Jensen's inequality, the *expected log price* drifts slower than μ even though the *expected price* grows at μ.
- We step daily with `dt = 1/252` (252 trading days/year) over a 1-year horizon.

A single path is built by cumulating these log-return increments and exponentiating.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

S0      = 100.0    # starting price
MU      = 0.08     # annual drift
SIGMA   = 0.20     # annual volatility
DAYS    = 252      # 1-year horizon in trading days
dt      = 1.0 / 252

# Per-step log-return distribution
mean_step = (MU - 0.5 * SIGMA**2) * dt
std_step  = SIGMA * np.sqrt(dt)

# Build ONE example path
z = np.random.normal(size=DAYS)
log_rets = mean_step + std_step * z
path = S0 * np.exp(np.cumsum(log_rets))
path = np.insert(path, 0, S0)   # prepend t=0

print("GBM parameters")
print(f"  S0={S0}  mu={MU}  sigma={SIGMA}  horizon={DAYS} days  dt={dt:.6f}")
print(f"  per-step log-return mean = {mean_step:.6f}, std = {std_step:.6f}")
print()
snap = pd.DataFrame({"day": [0, 21, 63, 126, 189, 252],
                     "price": [path[i] for i in [0, 21, 63, 126, 189, 252]]})
print("One simulated path (snapshots)")
print(snap.round(4).to_string(index=False))
print()
print(f"  Terminal price of this single path: {path[-1]:.4f}")

## 2. Simulate 10,000 paths

One path tells us almost nothing — it's a single draw from an infinite set of futures. We now simulate **10,000 independent paths** at once. The trick is to generate a `(n_paths × DAYS)` matrix of standard normals, scale into log returns, cumulate *along the time axis*, and exponentiate. Vectorizing with numpy keeps this well under a second. Each row is one possible future; each column is a point in time.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

S0, MU, SIGMA, DAYS = 100.0, 0.08, 0.20, 252
dt = 1.0 / 252
N_PATHS = 10000

mean_step = (MU - 0.5 * SIGMA**2) * dt
std_step  = SIGMA * np.sqrt(dt)

# (N_PATHS x DAYS) matrix of log-return increments
Z = np.random.normal(size=(N_PATHS, DAYS))
log_rets = mean_step + std_step * Z
log_price = np.cumsum(log_rets, axis=1)          # cumulate over time
paths = S0 * np.exp(log_price)                    # (N_PATHS x DAYS)
paths = np.hstack([np.full((N_PATHS, 1), S0), paths])  # prepend t=0 column

terminal = paths[:, -1]

print(f"Simulated {N_PATHS:,} GBM paths over {DAYS} days (seed=7)")
print(f"  paths matrix shape: {paths.shape}")
print()
# Theory check: E[S_T] = S0 * exp(mu*T) under GBM
expected_ST = S0 * np.exp(MU * 1.0)
print(f"  Theoretical E[S_T] = S0*exp(mu*T) = {expected_ST:.4f}")
print(f"  Simulated  mean S_T               = {terminal.mean():.4f}")
print(f"  Simulated  median S_T             = {np.median(terminal):.4f}")
print()
print("  (mean > median because the lognormal terminal distribution is right-skewed)")

## 3. The terminal price distribution

Because log returns are normal, the terminal price `S_T` is **lognormal** — bounded below by zero, with a long right tail. We summarize it with percentiles rather than just mean ± std, since the distribution is asymmetric. The 5th and 95th percentiles bracket a 90% confidence range for where the price lands after one year.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

S0, MU, SIGMA, DAYS = 100.0, 0.08, 0.20, 252
dt = 1.0 / 252
N_PATHS = 10000
mean_step = (MU - 0.5 * SIGMA**2) * dt
std_step  = SIGMA * np.sqrt(dt)

Z = np.random.normal(size=(N_PATHS, DAYS))
log_price = np.cumsum(mean_step + std_step * Z, axis=1)
terminal = S0 * np.exp(log_price[:, -1])

pcts = [5, 25, 50, 75, 95]
vals = np.percentile(terminal, pcts)

dist = pd.DataFrame({
    "statistic": ["mean", "std"] + [f"p{p}" for p in pcts],
    "price":     [terminal.mean(), terminal.std()] + list(vals),
})
dist["return_%"] = (dist["price"] / S0 - 1.0) * 100

print("Terminal price (S_T) distribution after 1 year")
print(dist.round(4).to_string(index=False))
print()
print(f"  90% of outcomes fall between {vals[0]:.2f} (p5) and {vals[-1]:.2f} (p95)")
print(f"  That is a return range of {(vals[0]/S0-1)*100:.1f}% to {(vals[-1]/S0-1)*100:.1f}%")

## 4. The percentile cone over time

Uncertainty doesn't arrive all at once — it **widens with the square root of time** (`σ√t`). If we slice the path matrix at several horizons and take percentiles at each, we get a *cone* that fans out from S0. This is the picture you see in fan charts: the median tracks the drift, while the p5/p95 envelope grows steadily wider the further out we look.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

S0, MU, SIGMA, DAYS = 100.0, 0.08, 0.20, 252
dt = 1.0 / 252
N_PATHS = 10000
mean_step = (MU - 0.5 * SIGMA**2) * dt
std_step  = SIGMA * np.sqrt(dt)

Z = np.random.normal(size=(N_PATHS, DAYS))
log_price = np.cumsum(mean_step + std_step * Z, axis=1)
paths = S0 * np.exp(log_price)
paths = np.hstack([np.full((N_PATHS, 1), S0), paths])   # include t=0

horizons = [0, 21, 63, 126, 189, 252]   # ~0, 1m, 3m, 6m, 9m, 12m
rows = []
for h in horizons:
    col = paths[:, h]
    rows.append({
        "day": h,
        "p5":  np.percentile(col, 5),
        "p25": np.percentile(col, 25),
        "p50": np.percentile(col, 50),
        "p75": np.percentile(col, 75),
        "p95": np.percentile(col, 95),
    })
cone = pd.DataFrame(rows)
cone["p95_minus_p5"] = cone["p95"] - cone["p5"]   # width of the cone

print("Percentile cone of price over time")
print(cone.round(3).to_string(index=False))
print()
print("The (p95 - p5) width grows with horizon ~ sqrt(time): uncertainty compounds.")

## 5. Value at Risk and downside probability

Two risk numbers fall straight out of the terminal distribution:

- **1-year 95% Value at Risk (VaR):** the loss not exceeded with 95% confidence. We take the 5th percentile of the *return* distribution; VaR is the magnitude of that loss. "There is a 5% chance of losing at least X."
- **Probability of ending below S0:** simply the fraction of paths whose terminal price is under the starting price — the chance the year is a net loss.

These are *historical-free* — they come entirely from the model and its parameters.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

S0, MU, SIGMA, DAYS = 100.0, 0.08, 0.20, 252
dt = 1.0 / 252
N_PATHS = 10000
mean_step = (MU - 0.5 * SIGMA**2) * dt
std_step  = SIGMA * np.sqrt(dt)

Z = np.random.normal(size=(N_PATHS, DAYS))
log_price = np.cumsum(mean_step + std_step * Z, axis=1)
terminal = S0 * np.exp(log_price[:, -1])

ret = terminal / S0 - 1.0                 # 1-year simple return per path

var95_ret = np.percentile(ret, 5)         # 5th percentile of returns
var95_loss = -var95_ret                    # VaR as a positive loss number
var95_dollars = S0 * var95_loss            # on a $S0 position

# Conditional VaR (expected shortfall): average loss in the worst 5%
cvar95 = -ret[ret <= var95_ret].mean()

prob_below_S0 = float(np.mean(terminal < S0))

risk = pd.DataFrame({
    "metric": ["95% VaR (return)", "95% VaR ($ on 100)", "95% CVaR / ES (return)",
               "P(end < S0)", "mean 1y return"],
    "value":  [f"{var95_loss*100:.2f}%", f"${var95_dollars:.2f}",
               f"{cvar95*100:.2f}%", f"{prob_below_S0*100:.2f}%",
               f"{ret.mean()*100:.2f}%"],
})
print("1-year risk metrics from the Monte Carlo distribution")
print(risk.to_string(index=False))
print()
print(f"Interpretation: with 95% confidence the 1y loss won't exceed {var95_loss*100:.1f}%;")
print(f"in the worst 5% of years the average loss is {cvar95*100:.1f}%.")

## 6. Drift vs diffusion — and the limits of GBM

The two forces in GBM pull in different ways. **Drift** (`μ`) is the deterministic engine — over long horizons it dominates and pushes the median upward. **Diffusion** (`σ`) is the random shock — it dominates short-horizon outcomes and is what creates the cone's width. Crucially, because shocks compound multiplicatively and volatility imposes the `−½σ²` drag, a high-vol asset can have a *median* return well below its drift even when its *mean* matches it.

**Where GBM breaks down:** it assumes constant volatility, normally distributed returns, and no jumps. Real markets show fat tails, volatility clustering, and sudden gaps — so true VaR is usually worse than GBM suggests. Still, GBM is the right first model: it is analytically tractable, captures drift + diffusion cleanly, and is the baseline every more-realistic model (stochastic vol, jump-diffusion) is measured against.


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
